# Deep & Two-Tower Recommenders

Companion notebook for the [Deep & Two-Tower lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/03-deep-and-two-tower).

We implement a **two-tower** scorer (user dot item), the **in-batch-negatives** contrastive loss
that trains it, and the **retrieval → ranking funnel** — showing how precomputed item embeddings
turn retrieval into a nearest-neighbor lookup. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## 1 — Two-tower scoring is a dot product in a shared space

The user tower and item tower each output a d-dim vector; the score is their dot product. Because
the towers are separate, **item embeddings can be precomputed once** and reused for every user.

In [ ]:
d = 8
n_items = 1000
item_emb = rng.normal(size=(n_items, d))                 # precomputed OFFLINE
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)

def user_tower(history_item_ids):
    """A toy user tower: average the embeddings of items the user engaged with."""
    u = item_emb[history_item_ids].mean(0)
    return u / np.linalg.norm(u)

u = user_tower([3, 17, 42])
scores = item_emb @ u                                    # one dot product per item
print('top-5 retrieved item ids:', np.argsort(-scores)[:5])
print('the user history items score high:', sorted(scores[[3,17,42]].round(2), reverse=True))

## 2 — In-batch-negatives contrastive loss

For a batch of B positive (user, item) pairs, each user's positive is its paired item and the
negatives are the *other* items in the batch. The loss is a softmax cross-entropy over the B×B
score matrix with the diagonal as the targets — B(B-1) negatives for free.

In [ ]:
def in_batch_loss(U, V):
    """U: (B,d) user embeddings, V: (B,d) their positive item embeddings."""
    logits = U @ V.T                                     # (B,B): row i vs all items in batch
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
    B = len(U)
    return -np.mean(np.log(p[np.arange(B), np.arange(B)] + 1e-9))

B = 6
# aligned pairs (each user embedding ~ its positive item) -> low loss
Vp = rng.normal(size=(B, d)); Vp /= np.linalg.norm(Vp, axis=1, keepdims=True)
Up = Vp + 0.05 * rng.normal(size=(B, d))                 # users near their positives
# random (mis-aligned) pairs -> high loss
Ur = rng.normal(size=(B, d))
print(f'loss, aligned pairs: {in_batch_loss(Up, Vp):.3f}  (should be low)')
print(f'loss, random  pairs: {in_batch_loss(Ur, Vp):.3f}  (should be higher)')

## 3 — The retrieval → ranking funnel

Retrieval cheaply shortlists candidates by dot product (recall). A heavier ranker then re-scores
only that shortlist with richer cross-features (precision). We simulate the two stages.

In [ ]:
def retrieve(u, item_emb, k):
    return np.argsort(-(item_emb @ u))[:k]               # cheap, scans all items

def ranker_score(u, item_ids, item_emb):
    # a heavier (here, nonlinear) scorer applied ONLY to the shortlist
    v = item_emb[item_ids]
    return (v @ u) ** 2 + 0.1 * v.sum(1)                 # toy cross-feature score

shortlist = retrieve(u, item_emb, k=50)                  # billions -> 50 (cheap)
ranked = shortlist[np.argsort(-ranker_score(u, shortlist, item_emb))][:10]
print('retrieval shortlist (first 8):', shortlist[:8])
print('final top-10 after ranking:   ', ranked)
print('ranking only re-scored 50 items, not all', n_items)

## ✏️ Your turn

**Exercise.** Implement `two_tower_score(u, v)` (the dot product of a user and item embedding) and
`retrieve_topk(u, item_emb, k)` returning the ids of the `k` highest-scoring items. This is exactly
the nearest-neighbor retrieval an ANN index accelerates in production.

In [ ]:
def two_tower_score(u, v):
    # TODO(you): score = dot product of the user and item embeddings
    return ...

def retrieve_topk(u, item_emb, k):
    # TODO(you): return the ids of the k items with the highest score against u
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.isclose(two_tower_score(u, item_emb[7]), u @ item_emb[7])
top = retrieve_topk(u, item_emb, 5)
assert list(top) == list(np.argsort(-(item_emb @ u))[:5])
# the top retrieved item scores at least as high as any other
assert two_tower_score(u, item_emb[top[0]]) >= two_tower_score(u, item_emb[123])
print('\u2713 two-tower scoring and top-k retrieval are correct')

<details>
<summary>Solution</summary>

```python
def two_tower_score(u, v):
    return u @ v

def retrieve_topk(u, item_emb, k):
    return np.argsort(-(item_emb @ u))[:k]
```

Because scoring is a dot product against precomputed item vectors, retrieval is exactly a
nearest-neighbor search — which an ANN index (HNSW/IVF) turns from O(items) into milliseconds at
billion-item scale.

</details>